# ทดลอง Extractor v2 — เห็นผลก่อนและหลังตรวจหลักฐาน

เลือก Python kernel ที่ติดตั้ง dependencies ของโปรเจกต์ แล้วรันจากบนลงล่าง

1. **ทดสอบตัวตรวจแบบไม่เรียก API** — มี output ตัวอย่างที่รันจริงเก็บไว้ด้านล่าง
2. **ทดสอบ Gemini จริง** — ป้อนข้อความ แล้วเทียบค่าที่โมเดลส่งกับค่าที่ผ่าน `ground()`
3. **ลองสนทนาจริงหลายรอบ** — ดูคำตอบ context และ `problem` ที่บันทึก

ใช้ข้อมูลสมมติเท่านั้น ส่วนที่เรียก Gemini มีค่าใช้จ่ายตามบัญชี ใช้ SQLite ในหน่วยความจำ ไม่ส่ง LINE
หลังแก้ไฟล์ Python ให้ Restart Kernel ก่อนรันใหม่ ชุดนี้ไม่ประเมินความเสี่ยงและไม่แยก `needs`


In [ ]:
from pathlib import Path
import sys
import json

BASE = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dspy-agent" / "app" / "extraction.py").is_file()), None)
if BASE is None:
    raise RuntimeError("เปิด notebook ภายในโฟลเดอร์ DSI490")
ROOT = BASE / "dspy-agent"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.extraction import Extractor, ground
from app.schemas import Extraction, IncidentData

print("พร้อมทดสอบ Extractor v2")
# หาก import ไม่ได้ ให้ติดตั้งครั้งแรก แล้ว Restart Kernel:
# %pip install -r {str(ROOT / "requirements-notebook.txt")}


## 1. ตรวจค่าที่โมเดลส่งผิดแบบไม่เรียก API

ตั้งใจจำลองผลสกัดที่ผิด เพื่อให้เห็นว่าตัวตรวจตัดอะไรทิ้ง ช่อง “ผลที่ควรได้” เป็นเกณฑ์เทส ไม่ใช่คำแนะนำทางการแพทย์


In [ ]:
checks = [
    {"name": "สองคน แต่โมเดลส่ง 99", "text": "มีเด็กสองคน",
     "raw": {"affected_people": {"children": 99}, "affected_people_evidence": {"children": "เด็กสองคน"}},
     "expected": None},
    {"name": "สิบสองคน ต้องไม่เหลือแค่ 2", "text": "มีเด็กสิบสองคน",
     "raw": {"affected_people": {"children": 2}, "affected_people_evidence": {"children": "เด็กสิบสองคน"}},
     "expected": None},
    {"name": "สิบสองคน ตรงกับ 12", "text": "มีเด็กสิบสองคน",
     "raw": {"affected_people": {"children": 12}, "affected_people_evidence": {"children": "เด็กสิบสองคน"}},
     "expected": 12},
    {"name": "เลขไทย ๑๒", "text": "มีเด็ก ๑๒ คน",
     "raw": {"affected_people": {"children": 12}, "affected_people_evidence": {"children": "เด็ก ๑๒ คน"}},
     "expected": 12},
    {"name": "evidence เป็นช่องว่าง", "text": "มีแผล",
     "raw": {"problem": "ข้อมูลที่โมเดลแต่ง", "problem_evidence": "   ", "context_complete": True},
     "expected": None},
]

check_results = []
for item in checks:
    before = Extraction.model_validate(item["raw"])
    after = ground(before, item["text"])
    if "affected_people" in item["raw"]:
        actual = after.affected_people.children
        ok = actual == item["expected"]
        comparison = {"ก่อนตรวจ": before.affected_people.children,
                      "หลังตรวจ": actual, "evidenceที่เหลือ": after.affected_people_evidence}
    else:
        ok = after.problem is None and not after.context_complete
        comparison = {"ก่อนตรวจ": before.problem, "หลังตรวจ": after.problem,
                      "context_complete": after.context_complete}
    check_results.append({"case": item["name"], "passed": ok})
    print("\n" + ("ผ่าน" if ok else "ไม่ผ่าน") + " | " + item["name"])
    print(json.dumps(comparison, ensure_ascii=False, indent=2))
print(f"\nสรุป: {sum(r['passed'] for r in check_results)}/{len(check_results)} เคสผ่าน")


## 2. ให้ Gemini สกัดจริง

อ่าน key จาก `.env` ที่มีอยู่ ไม่แสดง key ตั้ง `real_messages` เพื่อเพิ่มข้อความเองได้
แต่ละข้อความในส่วนนี้เป็นการสกัดอิสระ ไม่ได้ใช้ประวัติร่วมกัน ดูการสนทนาต่อเนื่องในหัวข้อ 3


In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=False)
load_dotenv(BASE / "lab" / "1" / ".env", override=False)
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("ตั้ง GEMINI_API_KEY หรือ GOOGLE_API_KEY ใน .env ก่อน")
MODEL = os.getenv("GEMINI_MODEL", "gemini/gemini-3.1-flash-lite")
dspy.configure(lm=dspy.LM(model=MODEL, api_key=api_key, cache=False, timeout=45, num_retries=0))
extractor = Extractor()
print("ใช้ LLM จริง:", MODEL)


In [ ]:
real_messages = [
    "น้ำท่วมบ้าน ออกไม่ได้ มีเด็กสองคน ขอเรือกับน้ำดื่ม",
    "มีเด็กสิบสองคนติดอยู่ที่โรงเรียน ต้องการรถอพยพ",
    "มีแผล",
    "ลูกมีไข้ตั้งแต่เมื่อคืน อายุ 5 ขวบ อยากปรึกษา",
]

extraction_results = []
for text in real_messages:
    print("\n" + "=" * 55)
    print("ผู้ใช้:", text, flush=True)
    try:
        raw = extractor(text, IncidentData(), None, "")
        checked = ground(raw, text)
        extraction_results.append({"text": text, "raw": raw.model_dump(mode="json"),
                                   "checked": checked.model_dump(mode="json")})
        print("LLM ก่อนตรวจ:")
        print(json.dumps(raw.model_dump(mode="json", exclude_none=True), ensure_ascii=False, indent=2))
        print("หลังตรวจ evidence:")
        print(json.dumps(checked.model_dump(mode="json", exclude_none=True), ensure_ascii=False, indent=2))
    except Exception as exc:
        extraction_results.append({"text": text, "error": type(exc).__name__})
        print("เรียกไม่สำเร็จ:", type(exc).__name__, "— ตรวจเครือข่าย model และ quota")


## ตัวช่วยอ่าน Database

อ่านเฉพาะผู้ใช้ทดลองที่เลือก ไม่อ่าน API key หรือ LINE ID จริง ช่อง JSON ถูกแปลงให้อ่านง่าย
Pipeline v4 ส่ง incident_priority เฉพาะ level และ reason ตาม Ruleset 1.1.0 ส่วน Risk Facts และ Audit อยู่ภายใน


In [ ]:
def database_snapshot(store, user_ref):
    """SELECT actual rows for one user. JSON TEXT is decoded only for display."""
    result = {}
    with store.transaction() as cursor:
        queries = {
            "users": ("SELECT * FROM users WHERE user_ref=?", (user_ref,)),
            "cases": ("SELECT * FROM cases WHERE user_ref=?", (user_ref,)),
            "case_raw_messages": ("SELECT * FROM case_raw_messages WHERE user_ref=? ORDER BY created_at", (user_ref,)),
        }
        for table in ("case_drafts", "case_slots", "case_personal_data", "case_versions",
                      "case_version_contacts", "case_events", "pipeline_confirmed_cases_v4", "pipeline_case_contacts"):
            queries[table] = (f"SELECT * FROM {table} WHERE case_id IN (SELECT case_id FROM cases WHERE user_ref=?)", (user_ref,))
        for table, (query, args) in queries.items():
            rows = store._exec(cursor, query, args)
            columns = [column[0] for column in rows.description]
            result[table] = []
            for values in rows.fetchall():
                row = dict(zip(columns, values))
                for key in ("payload", "incident"):
                    if isinstance(row.get(key), str):
                        row[key] = json.loads(row[key])
                result[table].append(row)
    return result


def show_database(snapshot):
    for table in ("cases", "case_drafts", "case_slots", "case_personal_data",
                  "case_versions", "case_version_contacts", "pipeline_confirmed_cases_v4"):
        print("\n" + "=" * 55 + "\n" + table)
        print(json.dumps(snapshot[table], ensure_ascii=False, indent=2))
    print("\nตารางอื่น เช่นข้อความและ events ดูได้จากตัวแปร snapshot โดยตรง")


## 3. คุยกับ Agent จริงต่อเนื่อง

เปลี่ยน `conversation` ได้ตามต้องการ แต่ละรอบแสดงคำตอบ สิ่งที่บันทึกใน `problem` และ context ครบหรือยัง
DB นี้แยกจากหน้าแชตอื่น ปิดท้ายเซลล์แล้วลบข้อมูลทดลองออกจากหน่วยความจำ


In [ ]:
from app.agent import Agent1
from app.store import Store
from app.knowledge import KnowledgeBase
from app.response import ResponseGenerator

conversation = [
    "มีแผล",
    "ที่ขา โดนของมีคมมาเมื่อคืน",
    "แผลเล็ก ยังแสบ อยากปรึกษาว่าควรดูแลอย่างไร",
]

trial_store = Store("sqlite:///:memory:")
trial_agent = Agent1(trial_store,
                     KnowledgeBase(ROOT / "app" / "knowledge_th.csv", allow_unreviewed=True),
                     generator=ResponseGenerator(use_llm=True))
conversation_results = []
try:
    trial_agent.handle("extraction_trial", postback="consent=yes")
    for text in conversation:
        print("\nผู้ใช้:", text, flush=True)
        try:
            output = trial_agent.handle("extraction_trial", text=text)
            case = trial_store.load_case(output.case_id) if output.case_id else None
            state = {"problem": case.incident.problem if case else None,
                     "context_complete": case.context_complete if case else None,
                     "pending_question": output.pending_question}
            conversation_results.append({"input": text, "output": output.model_dump(mode="json"), "state": state})
            print("บอท:", output.output_text)
            print("ข้อมูลที่เก็บ:", json.dumps(state, ensure_ascii=False))
        except Exception as exc:
            print("เรียกไม่สำเร็จ:", type(exc).__name__)
            break
finally:
    trial_database_snapshot = database_snapshot(trial_store, "extraction_trial")
    trial_store.close()


## 4. ดูข้อมูลที่เก็บจากบทสนทนาจริงด้านบน

- `case_drafts`: `problem` และรายละเอียดที่ยังแก้ไขได้
- `case_slots`: ความจำ คำถามค้าง และ context ครบหรือยัง
- `case_personal_data`: ข้อมูลติดต่อและพิกัด แยกจากข้อมูลที่ส่ง LLM
- `case_versions`: สำเนาข้อมูลหลังผู้ใช้กดยืนยัน
- `pipeline_confirmed_cases_v4`: สิ่งที่หลังบ้านอ่านได้จาก SQL view

ถ้ายังไม่ยืนยัน **versions และ Pipeline จะเป็น `[]`** เป็นพฤติกรรมที่ถูกต้อง


In [ ]:
if "trial_database_snapshot" not in globals():
    print("รันหัวข้อ 3 ก่อนเพื่อดูข้อมูลจากบทสนทนา Gemini จริง")
else:
    show_database(trial_database_snapshot)
# ดูข้อความดิบ: trial_database_snapshot["case_raw_messages"]
# ดู events: trial_database_snapshot["case_events"]


## 5. ตัวอย่างบันทึกครบและยืนยันแล้ว — ไม่เรียก API

ส่วนนี้ใช้ Extractor จำลองเพื่อสร้างข้อมูลสมมติ แต่บันทึกและอ่านผ่าน Store/SQLite จริง
ยืนยันผ่านปุ่มของ Agent จริง เปรียบเทียบก่อนกับหลังยืนยัน แล้วแสดง JSON ที่ Pipeline อ่านได้
แยก DB จากการคุยด้านบน ไม่ส่งข้อมูลออก LINE หรือฐานข้อมูล production


In [ ]:
from app.agent import Agent1
from app.store import Store
from app.knowledge import KnowledgeBase
from app.response import ResponseGenerator


def sample_extractor(text, draft, pending, summary):
    return Extraction(
        intents=["incident"], problem="น้ำท่วมบ้าน ออกไม่ได้ ต้องการเรือ มีผู้ใหญ่ 2 คน",
        problem_evidence=text, context_complete=True,
        affected_people={"other": 2}, affected_people_evidence={"other": "ผู้ใหญ่ 2 คน"},
    )


sample_store = Store("sqlite:///:memory:")
sample_agent = Agent1(sample_store, KnowledgeBase(), extractor=sample_extractor,
                      generator=ResponseGenerator(use_llm=False))
try:
    sample_agent.handle("database_demo", postback="consent=yes")
    sample_agent.handle("database_demo", text="น้ำท่วมบ้าน ออกไม่ได้ ต้องการเรือ มีผู้ใหญ่ 2 คน")
    sample_agent.handle("database_demo", location=(14.8936, 100.3967))
    sample_agent.handle("database_demo", text="สมชาย")
    sample_agent.handle("database_demo", text="ใจดี")
    summary = sample_agent.handle("database_demo", text="0812345678")  # เบอร์สมมติ
    print("สรุปที่ผู้ใช้ต้องตรวจ:\n" + summary.output_text)
    print("\nPipeline ก่อนยืนยัน:", sample_store.pipeline_rows())
    before_confirmation = database_snapshot(sample_store, "database_demo")
    confirm = next(button.data for button in summary.quick_replies
                   if button.data and button.data.startswith("confirm="))
    sample_agent.handle("database_demo", postback=confirm)
    confirmed_database_snapshot = database_snapshot(sample_store, "database_demo")
    confirmed_pipeline = sample_store.pipeline_rows()
    print("\nPipeline หลังยืนยัน (ผลจริงจาก Store):")
    print(json.dumps(confirmed_pipeline, ensure_ascii=False, indent=2))
    print("\nชื่อ/เบอร์แยกไว้ที่:")
    print(json.dumps(confirmed_database_snapshot["case_version_contacts"], ensure_ascii=False, indent=2))
finally:
    sample_store.close()


In [ ]:
# เปิดดูทุกตารางของตัวอย่างที่ยืนยันแล้ว
show_database(confirmed_database_snapshot)
